# PSA_ReadData: シンチレータ(PSA)の生データの読み込みとイベントの同期

SMILE-3 のシンチレータ(HA)は、モジュールごとに PSA ボードが 1 枚あり、MPPC 6 個(ch 0〜5)の波形を記録します。
`PSA_ReadData` は、ランのディレクトリにある**全ボードの生データ `ha<NN>_<番号>.dat` を読み、トリガーごとに 1 イベントにまとめます**。
ファイルの中身の読み取りは `PSA_Board`(`source/modules/PSA_ReadData/PSA_Board.cxx`)で、
`analysis_sm3_FM/PSA_analysis` の `psa_decode.py` と同じ読み方です。

| | |
|---|---|
| 入力 | `ha<NN>_<番号>.dat`。ボードごとに番号の順につなげて読む(`.dat` が無い番号は `.dat.tmp`。storage が落ちて閉じられなかったファイル) |
| 後ろのモジュールに渡すもの | ボードごとに: このイベントにあるか(`Present`)、トリガーカウンタ、時刻、ヒットのある ch、波形(50 サンプル × 8 本)、温度センサーの生の値、飽和 |
| ROOT ファイルに書くもの | `PSA_ReadData/HitPattern`(ボード × ch のヒットの数) |

### イベントのまとめ方

| チェーン | まとめ方 |
|---|---|
| `ReadData`(TPC)が無い(PSA だけ。HA_cal など) | 全ボードの「次のフレーム」のうち、トリガーカウンタが一番小さいものを 1 イベントにする。同じカウンタのフレームを持つボードだけが、そのイベントに「ある」 |
| `ReadData` が前にあり `sync_tpc: true` | `ReadData` のイベントのトリガーカウンタに合わせる。それより小さいカウンタのフレームは読み飛ばし、大きいものは次のイベントまで取っておく |

PSA だけのときにトリガーカウンタが戻ったら(ランの途中のリセットなど)、`epoch` を 1 つ増やして、続きを別のイベントとして扱います。

### フレーム(1 ボード・1 トリガー)の形

ビッグエンディアンです。ヒットの無いボードも、ヘッダ・温度・フッタだけの 40 バイトのフレームを書くことがあります。

| 位置 | 中身 |
|---|---|
| 0x00 | ヘッダ `eb 90 19 64` |
| 0x04 | トリガーカウンタ(TCU、全ボードで共通) |
| 0x08 | クロック(1 MHz) |
| 0x0D | 2 バイト: ビット 6〜11 がデータのある ADC(ADC j ならビット 5 − j) |
| 0x14 | 32 ビット × 3: 温度センサー 6 個の生の値(14 ビットずつ) |
| 0x24〜 | 16 バイトのブロック。データのある ADC 1 個につき 50 個(サンプル 1〜50)。1 ブロックに ADC の番号、サンプル番号、14 ビットの値 8 個(4 隅の HG、4 隅の LG) |
| 最後 | フッタ `75 50 49 43`(`uPIC`) |

長さは 40 + 800 × (データのある ADC の数) バイトです。ch の番号は **ch = 5 − j**(ADC j)で、`psa_decode.py` の `.h5` の `ch`、
`peak_search.py` の `position_haNN.csv` の `ch` と同じです。

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CACHE = REPO / "tutorials" / "cache"   # チェーンの出力の置き場(チュートリアル間で共有、git には入れない)
CACHE.mkdir(exist_ok=True)

def th2(h):
    """ROOT の TH2 -> (中身 [x のビン, y のビン], x のビンの端, y のビンの端)"""
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    a = np.frombuffer(h.GetArray(), dtype=np.float64, count=(nx + 2) * (ny + 2)).reshape(ny + 2, nx + 2)
    ex = np.array([h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)])
    ey = np.array([h.GetYaxis().GetBinLowEdge(i) for i in range(1, ny + 2)])
    return a[1:-1, 1:-1].T.copy(), ex, ey

def show_params(config, module):
    """設定ファイルの、あるモジュールのパラメータを、説明(_doc)と単位(_units)つきで表示する"""
    p = json.load(open(REPO / "config" / f"{config}.json"))["parameters"][module]
    doc, units = p.get("_doc", {}), p.get("_units", {})
    for k, v in p.items():
        if k.startswith("_"):
            continue
        u = f" [{units[k]}]" if k in units else ""
        print(f"{k} = {json.dumps(v, ensure_ascii=False)}{u}\n    {doc.get(k, '')}")

## 1. データの準備

2026-11-03 の UVSOR での **HA_cal のラン `20261103_run01`**(PSA だけ、14 ボード、約 840 MB)を使います。
位置の較正(`calib/psa/position_haNN.csv`)を作ったときと同じ日の較正データです。
データは解析用の計算機の D ドライブ(`/mnt/d/UVSOR2026_late`)にあります。
ほかの場所にあるときは、環境変数 `SMILE3_PSA_CAL_DATA` に `UVSOR2026_late` に当たるディレクトリを設定してください。

1 つのランのボードは、2 つの storage に分かれて書かれています。

| storage | ディレクトリ | PSA のボード |
|---|---|---|
| storage_g1 | `killua/<ラン>` | ha00〜ha05 |
| storage_g0 | `calltt/<ラン>` | ha06〜ha16 |

`run_chain.py` には、片方を `--data-dir`、もう片方を `PSA_ReadData.extra_data_dirs` で渡します。

In [ ]:
DATA = Path(os.environ.get("SMILE3_PSA_CAL_DATA", "/mnt/d/UVSOR2026_late"))
RUN = "20261103_run01"
RUN_DIRS = [DATA / "killua" / RUN, DATA / "calltt" / RUN]
for d in RUN_DIRS:
    print("OK     " if d.is_dir() else "MISSING", d)

ボードごとのファイルの数と大きさです。ボードによって大きさがかなり違います(ヒットの多いボードほど大きい)。

In [ ]:
import re
rows = []
for d in RUN_DIRS:
    for f in sorted(d.glob("ha*_*.dat*")):
        m = re.match(r"ha(\d{2})_(\d{5})\.dat(\.tmp)?$", f.name)
        if m:
            rows.append((int(m.group(1)), d.parent.name, f.stat().st_size, bool(m.group(3))))
for b in sorted({r[0] for r in rows}):
    rs = [r for r in rows if r[0] == b]
    print(f"ha{b:02d} ({rs[0][1]}): {len(rs):3d} files, {sum(r[2] for r in rs) / 1e6:7.1f} MB, .dat.tmp {sum(r[3] for r in rs)}")

## 2. チェーンを流す

設定ファイル `smile3_psa`(`PSA_ReadData → PSA_CalcFADC → PSA_CenterGravity → PSA_Calibration → PSA_TreeWriter`)で、ラン全体を流します(40 秒ほど)。
ターミナルでは、リポジトリの `run_script` で次を実行するのと同じです。

```bash
python3 run_chain.py smile3_psa -o psa_20261103_run01.root \
    --data-dir /mnt/d/UVSOR2026_late/killua/20261103_run01 \
    --set 'PSA_ReadData.extra_data_dirs=["/mnt/d/UVSOR2026_late/calltt/20261103_run01"]'
```

出力は `tutorials/cache/psa_cal_<ラン>.root` に置き、既にあれば流し直しません(PSA のチュートリアルと `4_MakeCalibration/PSA_Energy` で共有)。
`calib/psa` の較正ファイルを変えたときは、このファイルを消して流し直してください。

In [ ]:
def run_chain(out, *args):
    """run_chain.py を実行する(出力が既にあれば何もしない)"""
    cmd = [sys.executable, str(RUN_CHAIN), "smile3_psa", "-o", str(out), "--data-dir", str(RUN_DIRS[0]),
           "--set", "PSA_ReadData.extra_data_dirs=" + json.dumps([str(RUN_DIRS[1])]), *args]
    print(" ".join(cmd))
    if not out.exists():
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
        r.check_returncode()

OUT = CACHE / f"psa_cal_{RUN}.root"   # 今の calib/psa(位置・温度ゲイン・エネルギー)で流したもの
run_chain(OUT)

## 3. どのボード・ch にヒットがあったか

`PSA_ReadData/HitPattern` は、ボード × ch ごとのヒットの数です(`save_hit_pattern: true`)。
色は対数です。ha04 は ch1 だけヒットが多く(ほかのボードの ch の 7〜8 倍)、ha04 のほかの ch は少なくなっています。
2026-11-02 のランでも ha04 の ch1 だけが飛び抜けて多かったので、この ch に何かある(ノイズなど)のかもしれません。

In [ ]:
f = ROOT.TFile(str(OUT))
hp, ex, ey = th2(f.Get("PSA_ReadData/HitPattern"))
fig, ax = plt.subplots(figsize=(10, 3.5))
m = ax.pcolormesh(ex, ey, hp.T, norm=LogNorm(vmin=1), shading="flat")
fig.colorbar(m, ax=ax, label="hits")
ax.set_xlabel("HA board"); ax.set_ylabel("ch"); ax.set_xticks(range(len(ex) - 1))
ax.set_title(f"PSA_ReadData/HitPattern ({RUN})")
plt.show()
for b in np.where(hp.sum(axis=1) > 0)[0]:
    print(f"ha{b:02d}: hits {int(hp[b].sum()):7d}  per ch {hp[b].astype(int)}")

## 4. 1 イベントのヒットの数

`PSA_TreeWriter/psa` は 1 ヒット 1 行のツリーで、`sevt_id` が `PSA_ReadData` のイベントの番号(0 から)です。
同じ `sevt_id` の行が、同じトリガーのヒットです。ツリーの列の説明は [6_PSA_TreeWriter](../6_PSA_TreeWriter/PSA_TreeWriter.ipynb) にあります。

1 イベントに何ヒット・何ボードあるかを数えます。

In [ ]:
def read_psa(path, columns):
    """PSA_TreeWriter/psa の列を numpy の配列で読む(short の列は int に直す)"""
    d = ROOT.RDataFrame("PSA_TreeWriter/psa", str(path)).AsNumpy(columns)
    out = {}
    for k, v in d.items():
        if v.dtype == object and len(v) and hasattr(v[0], "__len__"):
            out[k] = np.stack([np.asarray(x) for x in v])   # 配列の列 -> 2 次元
        elif v.dtype == object:
            out[k] = v.astype(int)
        else:
            out[k] = v
    return out

In [ ]:
t = read_psa(OUT, ["sevt_id", "trig", "epoch", "board", "ch", "time", "hitnum"])
ev, n_hit = np.unique(t["sevt_id"], return_counts=True)
_, first = np.unique(t["sevt_id"], return_index=True)
n_board = np.array([len(set(x)) for x in np.split(t["board"][np.argsort(t["sevt_id"], kind="stable")], np.cumsum(n_hit)[:-1])])
print("rows (hits):", len(t["trig"]), " events with hits:", len(ev), " epochs:", np.unique(t["epoch"]))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(n_hit, bins=np.arange(0.5, 10.5), log=True); ax[0].set_xlabel("hits in the event"); ax[0].set_ylabel("events")
ax[1].hist(n_board, bins=np.arange(0.5, 10.5), log=True); ax[1].set_xlabel("boards with hits in the event"); ax[1].set_ylabel("events")
fig.tight_layout(); plt.show()
print("hits per event:", {k: int(v) for k, v in zip(*np.unique(n_hit, return_counts=True))})

## 5. トリガーカウンタと時刻

`time` はボードごとの時刻 [s] で、そのボードの最初のフレームのクロックからの時間です(32 ビットのクロックの一周りはつないでいる)。
`trig` は全ボードで共通なので、ボードをまたいでイベントを合わせるのに使います。

下の図は、ボードごとのヒットの数の時間変化です。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(0, t["time"].max(), 85)
for b in np.unique(t["board"]):
    sel = t["board"] == b
    ax.hist(t["time"][sel], bins=bins, histtype="step", label=f"ha{b:02d}")
ax.set_yscale("log"); ax.set_xlabel("time [s]"); ax.set_ylabel("hits / bin"); ax.legend(ncol=7, fontsize=8)
plt.show()
print("trig range:", t["trig"].min(), "-", t["trig"].max())

## 6. パラメータ

`config/smile3_psa.json` の `PSA_ReadData` のパラメータと、その説明です。

In [ ]:
show_params("smile3_psa", "PSA_ReadData")